# 06 - Reserved Slot Visual Walkthrough

This notebook shows what the booking policy is doing to the appointment calendar. It uses one editable, small scenario so the allocation is visible slot by slot.

The point is not to estimate performance. The point is to see how pooled FCFS and strict reservation allocate the same arrivals.


In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd


def find_repo_dir(start: Path) -> Path:
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "simulation" / "engine.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from simulation.engine import ClinicAppointmentSimulation
from simulation.model import PatientClassParams, SimulationConfig, ThresholdRule

plt.style.use("default")


## Editable Visual Scenario

Keep `slots_per_day` small if you want the slot grid to remain readable. The default arrival order is designed to show the difference between pooled FCFS and strict reservation: Class 2 demand exceeds the general pool while Class 1 does not fill all protected slots.


In [ ]:
VISUAL_SCENARIO = {
    "slots_per_day": 8,
    "horizon_days": 1,
    "reserved_class_id": 1,
    "reserved_slots_per_day": 3,
    "arrival_order": [2, 2, 2, 2, 2, 2, 1],
}

pd.Series(VISUAL_SCENARIO, name="value").to_frame()


## Run The Same Arrivals Under Each Policy

Balking, cancellation, and no-show probabilities are set to zero here. That isolates allocation logic: who gets a slot, which pool they use, and who receives no offer.

In [ ]:
ZERO_RULE = ThresholdRule(threshold=0, low=0.0, high=0.0)


def visual_config(policy: str) -> SimulationConfig:
    class_ids = sorted(set(VISUAL_SCENARIO["arrival_order"]) | {VISUAL_SCENARIO["reserved_class_id"]})
    classes = {
        class_id: PatientClassParams(
            class_id=class_id,
            lambda_per_day=0.0,
            balk_prob=ZERO_RULE,
            cancel_prob=0.0,
            no_show_prob=ZERO_RULE,
        )
        for class_id in class_ids
    }
    reserved = policy != "pooled_fcfs"
    return SimulationConfig(
        slots_per_day=VISUAL_SCENARIO["slots_per_day"],
        horizon_days=VISUAL_SCENARIO["horizon_days"],
        burn_in_days=0,
        measure_days=1,
        cooldown_days=0,
        classes=classes,
        reserved_class_id=VISUAL_SCENARIO["reserved_class_id"] if reserved else None,
        reserved_slots_per_day=VISUAL_SCENARIO["reserved_slots_per_day"] if reserved else 0,
    )


policy_order = ["pooled_fcfs", "strict_reservation"]
policy_runs = {}

for policy in policy_order:
    config = visual_config(policy)
    sim = ClinicAppointmentSimulation(config)
    sim.process_daily_arrivals(VISUAL_SCENARIO["arrival_order"], track_patients=True)
    policy_runs[policy] = {"config": config, "sim": sim}

pd.DataFrame(
    [
        {
            "policy": policy,
            "slots_per_day": run["config"].slots_per_day,
            "reserved_slots": run["config"].reserved_slots_per_day,
            "general_slots": run["config"].slots_per_day - run["config"].reserved_slots_per_day,
        }
        for policy, run in policy_runs.items()
    ]
)


## Allocation Table

Each row is a pool on one residual day. `C1` and `C2` are booked patients; `empty` is unused capacity.

In [ ]:
def slot_labels(bookings, *, reserved_slot: bool, capacity: int) -> list[str]:
    labels = [f"C{booking.patient_class}" for booking in bookings if booking.reserved_slot == reserved_slot]
    labels.extend(["empty"] * (capacity - len(labels)))
    return labels


allocation_rows = []
for policy, run in policy_runs.items():
    config = run["config"]
    sim = run["sim"]
    for r, bookings in enumerate(sim.calendar):
        if config.reserved_slots_per_day > 0:
            pools = [
                ("reserved", True, config.reserved_slots_per_day),
                ("general", False, config.slots_per_day - config.reserved_slots_per_day),
            ]
        else:
            pools = [("pooled", False, config.slots_per_day)]
        for pool, reserved_slot, capacity in pools:
            labels = slot_labels(bookings, reserved_slot=reserved_slot, capacity=capacity)
            allocation_rows.append(
                {
                    "policy": policy,
                    "r": r,
                    "pool": pool,
                    "occupied": capacity - labels.count("empty"),
                    "capacity": capacity,
                    "pool_utilization": (capacity - labels.count("empty")) / capacity if capacity else 0.0,
                    **{f"slot_{idx + 1}": label for idx, label in enumerate(labels)},
                }
            )

allocation_df = pd.DataFrame(allocation_rows)
allocation_df


## Calendar Grid

The grid makes the policy difference visible. Strict reservation leaves protected capacity empty when Class 1 does not use it, while pooled FCFS keeps all capacity shared.


In [ ]:
COLORS = {
    "C1": "#2563eb",
    "C2": "#dc2626",
    "empty_reserved": "#dbeafe",
    "empty_general": "#f3f4f6",
    "empty_pooled": "#f3f4f6",
}


def labels_by_day_and_pool(config: SimulationConfig, sim: ClinicAppointmentSimulation):
    rows = []
    if config.reserved_slots_per_day > 0:
        pool_specs = [
            ("reserved", True, config.reserved_slots_per_day),
            ("general", False, config.slots_per_day - config.reserved_slots_per_day),
        ]
    else:
        pool_specs = [("pooled", False, config.slots_per_day)]

    for pool, reserved_slot, capacity in pool_specs:
        for slot_idx in range(capacity):
            row = {"pool": pool, "slot": slot_idx + 1, "labels": []}
            for bookings in sim.calendar:
                labels = slot_labels(bookings, reserved_slot=reserved_slot, capacity=capacity)
                row["labels"].append(labels[slot_idx])
            rows.append(row)
    return rows


def draw_calendar(ax, policy: str, config: SimulationConfig, sim: ClinicAppointmentSimulation) -> None:
    rows = labels_by_day_and_pool(config, sim)
    for y, row in enumerate(rows):
        for r, label in enumerate(row["labels"]):
            if label == "empty":
                color = COLORS[f"empty_{row['pool']}"]
                text_color = "#6b7280"
            else:
                color = COLORS.get(label, "#9ca3af")
                text_color = "white"
            ax.add_patch(Rectangle((r, y), 1, 1, facecolor=color, edgecolor="white", linewidth=1.2))
            ax.text(r + 0.5, y + 0.5, label, ha="center", va="center", fontsize=9, color=text_color)

    ax.set_xlim(0, config.horizon_days)
    ax.set_ylim(0, len(rows))
    ax.invert_yaxis()
    ax.set_xticks([r + 0.5 for r in range(config.horizon_days)])
    ax.set_xticklabels([f"r={r}" for r in range(config.horizon_days)])
    ax.set_yticks([idx + 0.5 for idx in range(len(rows))])
    ax.set_yticklabels([f"{row['pool']} {row['slot']}" for row in rows])
    ax.set_title(policy.replace("_", " ").title())
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)


fig, axes = plt.subplots(len(policy_order), 1, figsize=(10, 3.2 * len(policy_order)))
if len(policy_order) == 1:
    axes = [axes]

for ax, policy in zip(axes, policy_order):
    run = policy_runs[policy]
    draw_calendar(ax, policy, run["config"], run["sim"])

fig.tight_layout()


## Outcomes And Utilization

This table shows how the same arrivals flow through each policy. Because no-show probability is zero in this visual scenario, booked slots would become served slots if service were resolved immediately.

In [ ]:
outcome_rows = []
for policy, run in policy_runs.items():
    config = run["config"]
    sim = run["sim"]
    total_booked = sum(metrics.booked for metrics in sim.class_metrics.values())
    total_balked = sum(metrics.balked for metrics in sim.class_metrics.values())
    total_no_offer = sum(metrics.no_offer for metrics in sim.class_metrics.values())
    total_arrivals = sum(metrics.arrivals for metrics in sim.class_metrics.values())
    reserved_booked = sum(
        booking.reserved_slot
        for day in sim.calendar
        for booking in day
    )
    general_booked = total_booked - reserved_booked
    outcome_rows.append(
        {
            "policy": policy,
            "arrivals": total_arrivals,
            "booked": total_booked,
            "balked": total_balked,
            "no_offer": total_no_offer,
            "reserved_booked": reserved_booked,
            "general_or_pooled_booked": general_booked,
            "calendar_capacity": config.slots_per_day * config.horizon_days,
            "booking_utilization": total_booked / (config.slots_per_day * config.horizon_days),
        }
    )

outcome_df = pd.DataFrame(outcome_rows)
outcome_df


In [ ]:
class_flow_rows = []
for policy, run in policy_runs.items():
    for class_id, metrics in run["sim"].class_metrics.items():
        class_flow_rows.append(
            {
                "policy": policy,
                "class_id": class_id,
                "arrivals": metrics.arrivals,
                "booked": metrics.booked,
                "balked": metrics.balked,
                "no_offer": metrics.no_offer,
            }
        )

pd.DataFrame(class_flow_rows)


## How To Read This

- Pooled FCFS does not distinguish reserved and general capacity.
- Strict reservation protects Class 1 slots even when Class 2 is waiting.

Change `arrival_order`, `slots_per_day`, and `reserved_slots_per_day` in the visual scenario cell to stress different mechanics.
